### 데이터 분석 컨설팅: 도시계획과

In [48]:
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
pd.set_option("display.max_rows", None)
import warnings
warnings.filterwarnings('ignore')

GPKG_PATH = r"02_자료수집\\2026-018_도시계획과.gpkg"
OUTPUT_PATH = r"03_집계결과\\"
ten_sectors = gpd.read_file(GPKG_PATH, layer="10개_중심지_생활권")
gugun = gpd.read_file(GPKG_PATH, layer="NGII_SIG_26")
bdong = gpd.read_file(GPKG_PATH, layer="NGII_EMD_26")
역세권350 = gpd.read_file(GPKG_PATH, layer="역세권350_74EA")
역세권500 = gpd.read_file(GPKG_PATH, layer="역세권500_58EA")

#### 1. 건축물

1. 건축물 연면적

In [ ]:

# 10개 중심지 생활권 별 건축물 연면적
def sectors_influence_area(gdf2):
    values = []
    for year in tqdm(range(2013, 2026)):
        gdf = gpd.read_file(GPKG_PATH, layer=f"GIS건물통합정보{str(year)}C")
        gdf['기준연도'] = gdf['데이터기준일자'].astype(str).str[:4] + "년"
        sjoined = gpd.sjoin(gdf2, gdf, how="inner", predicate="intersects")
        values.append(sjoined.pivot_table(index='기준연도', columns='CNAME', values='연면적', aggfunc='sum', fill_value=0).round(3).reset_index())
    concated = pd.concat(values, ignore_index=True)
    return concated

# 부산시 전체(구군별) 건축물 연면적
def gugun_influence_area(gdf2):
    values = []
    for year in tqdm(range(2019, 2026)):
        gdf = gpd.read_file(GPKG_PATH, layer=f"GIS건물통합정보{str(year)}C")
        gdf['기준연도'] = gdf['데이터기준일자'].astype(str).str[:4] + "년"
        sjoined = gpd.sjoin(gdf2, gdf, how="inner", predicate="intersects")
        values.append(sjoined.pivot_table(index='기준연도', columns='SIG_KOR_NM', values='연면적', aggfunc='sum', fill_value=0).round(3).reset_index())
    concated = pd.concat(values, ignore_index=True)
    concated = concated[['기준연도','중구','서구','동구','영도구','부산진구','동래구','남구','북구',
                               '해운대구','사하구','금정구','강서구','연제구','수영구','사상구','기장군']]
    return concated

# 132개 역세권 건축물 연면적
def station_influence_area(gdf2):
    values = []
    for year in tqdm(range(2019, 2026)):
        gdf = gpd.read_file(GPKG_PATH, layer=f"GIS건물통합정보{str(year)}C")
        gdf['기준연도'] = gdf['데이터기준일자'].astype(str).str[:4] + "년"
        sjoined = gpd.sjoin(gdf2, gdf, how="inner", predicate="intersects")
        values.append(sjoined.pivot_table(index=['기준연도','역세권'], columns='역명', values='연면적', aggfunc='sum', fill_value=0).round(3).reset_index())
    concated = pd.concat(values, ignore_index=True)
    return concated

# sectors_influence_area(ten_sectors).to_excel(OUTPUT_PATH + "1_기준연도_10개 중심지 생활권_건축물 연면적 현황.xlsx", engine="openpyxl", index=False)
gugun_influence_area(gugun).to_excel(OUTPUT_PATH + "1_기준연도_구군_건축물 연면적 현황.xlsx", engine="openpyxl", index=False)
station_influence_area(역세권350).to_excel(OUTPUT_PATH + "1_기준연도_역세권350_건축물 연면적 현황.xlsx", engine="openpyxl", index=False)
station_influence_area(역세권500).to_excel(OUTPUT_PATH + "1_기준연도_역세권500_건축물 연면적 현황.xlsx", engine="openpyxl", index=False)


100%|██████████| 7/7 [01:00<00:00,  8.58s/it]


2. 건축물용도별 연면적

In [52]:
buld_type = {
    '주거': ['공동주택', '단독주택','다가구주택','다세대주택'],
    '근생': ['근린생활시설','제1종근린생활시설'],
    '식음': ['제2종근린생활시설','위락시설'],
    '문화/교육/복지(의료)': ['문화및집회시설', '교육연구시설', '복지시설', '교육연구및복지시설', '의료시설'],
    '숙박': ['숙박시설'], '판매': ['판매시설','판매및영업시설'], '업무': ['업무시설','사무소'],
    '기타': ['공장', '제조업소', '창고', '종교', '자동차관련시설', '운수시설', '위험물저장및처리시설', '운동시설', 
             '수련시설', '관광휴게시설', '동식물관련시설', '분뇨·쓰레기처리시설', '교정및군사시설', '방소통신시설', 
	         '장례식장', '발전시설', '공공용시설', '묘지관련시설', None]}
name_to_group = {name: group for group, names in buld_type.items() for name in names}

# 10개 중심지 생활권별 건축물용도별 연면적
def sectors_buld_type_area(gdf2):
    values = []
    for year in tqdm(range(2019, 2026)):
        gdf1 = gpd.read_file(GPKG_PATH, layer=f"GIS건물통합정보{year}C")
        gdf1 = gdf1[gdf1['데이터기준일자'].astype(str).str[:4] == str(year)]
        gdf1['기준연도'] = gdf1['데이터기준일자'].astype(str).str[:4] + "년"
        gdf1['구분'] = gdf1['건축물용도명'].map(name_to_group)
        sjoined = gpd.sjoin(gdf2, gdf1, how="inner", predicate="intersects")
        values.append(sjoined.pivot_table(index=['기준연도','구분'], columns='CNAME', values='연면적', aggfunc='sum', fill_value=0).round().reset_index())
    concated = pd.concat(values, ignore_index=True)
    return concated

# 구군별 건축물용도별 연면적
def gugun_buld_type_area(gdf2):
    values = []
    for year in tqdm(range(2019, 2026)):
        gdf1 = gpd.read_file(GPKG_PATH, layer=f"GIS건물통합정보{year}C")
        gdf1 = gdf1[gdf1['데이터기준일자'].astype(str).str[:4] == str(year)]
        gdf1['기준연도'] = gdf1['데이터기준일자'].astype(str).str[:4] + "년"
        gdf1['구분'] = gdf1['건축물용도명'].map(name_to_group)
        sjoined = gpd.sjoin(gdf2, gdf1, how="inner", predicate="intersects")
        values.append(sjoined.pivot_table(index=['기준연도','구분'], columns='SIG_KOR_NM', 
                                          values='연면적', aggfunc='sum', fill_value=0).round().reset_index())
    concated = pd.concat(values, ignore_index=True)
    concated = concated[['기준연도','구분','중구','서구','동구','영도구','부산진구','동래구','남구','북구',
                               '해운대구','사하구','금정구','강서구','연제구','수영구','사상구','기장군']]
    return concated

# 역세권 건축물용도별 연면적
def station_buld_type_area(gdf2):
    values = []
    for year in tqdm(range(2019, 2026)):
        gdf1 = gpd.read_file(GPKG_PATH, layer=f"GIS건물통합정보{year}C")
        gdf1 = gdf1[gdf1['데이터기준일자'].astype(str).str[:4] == str(year)]
        gdf1['기준연도'] = gdf1['데이터기준일자'].astype(str).str[:4] + "년"
        gdf1['구분'] = gdf1['건축물용도명'].map(name_to_group)
        sjoined = gpd.sjoin(gdf2, gdf1, how="inner", predicate="intersects")
        values.append(sjoined.pivot_table(index=['기준연도','구분','역세권'], columns='역명',
                                          values='연면적', aggfunc='sum', fill_value=0).round().reset_index())
    concated = pd.concat(values, ignore_index=True)
    return concated

sectors_buld_type_area(ten_sectors).to_excel(OUTPUT_PATH + "1_용도구분_10개 중심지 생활권_건축물 연면적 현황.xlsx", engine="openpyxl", index=False)
gugun_buld_type_area(gugun).to_excel(OUTPUT_PATH + "1_용도구분_구군_건축물 연면적 현황.xlsx", engine="openpyxl", index=False)
station_buld_type_area(역세권350).to_excel(OUTPUT_PATH + "1_용도구분_역세권350_건축물 연면적 현황.xlsx", engine="openpyxl", index=False)
station_buld_type_area(역세권500).to_excel(OUTPUT_PATH + "1_용도구분_역세권500_건축물 연면적 현황.xlsx", engine="openpyxl", index=False)

100%|██████████| 7/7 [00:45<00:00,  6.51s/it]


3. 허가 연면적
- 공공데이터 포털 국토교통부 건축인허가 기본개요 수집데이터 활용

In [ ]:
import pandas as pd

DATA_PATH = r"02_자료수집\\01_건축물정보\\"

data = pd.read_excel(DATA_PATH + "국토교통부_건축인허가_기본개요.xlsx", engine="openpyxl")
# print(data.shape) 
# (196122, 45)
data = data.dropna(subset=['건축허가일'])
data['건축허가일'] = data['건축허가일'].astype(str)
data['허가연도'] = data['건축허가일'].str[:4] + "년"
filtered = data[data['허가연도'].str.contains('2019|2020|2021|2022|2023|2024|2025')]

# 대지위치 기준으로 좌표추출
# data[['순번','대지위치']].to_excel(DATA_PATH + "국토교통부_건축인허가_기본개요_대지위치.xlsx", engine="openpyxl", index=False)

# 좌표정보가 포함된 파일을 병합
addr_to_coord = pd.read_excel(DATA_PATH + "주소 좌표 변환 목록_20260909155851.xlsx", engine="openpyxl")
merged = pd.merge(filtered, addr_to_coord[['주소','위도','경도','정제결과']], left_on="대지위치", right_on="주소", how="left")

# 좌표값을 이용하여 GeoDataFrame으로 변환
gbase = gpd.GeoDataFrame(merged, geometry=gpd.points_from_xy(merged['경도'], merged['위도']), crs=4326).to_crs(5179)
gbase['X'] = gbase.geometry.x.round(3)
gbase['Y'] = gbase.geometry.y.round(3)
gbase.drop(['위도','경도'], axis=1, inplace=True)

# 10대 중심지 생활권 허가연면적 합계
def sector_hub_baseinfo(gdf1, gdf2):
    sjoined = gpd.sjoin(gdf2, gdf1, how="inner", predicate="intersects")
    pivoted = sjoined.pivot_table(index='허가연도', columns='CNAME', values='연면적(㎡)', aggfunc='sum', fill_value=0).round(3).reset_index()
    return pivoted

# 구군별 허가연면적 합계
pivoted = gbase.pivot_table(index='허가연도', columns='시군구명', values='연면적(㎡)', aggfunc='sum', fill_value=0).round(3).reset_index()
pivoted = pivoted[['허가연도','중구','서구','동구','영도구','부산진구','동래구','남구','북구',
                         '해운대구','사하구','금정구','강서구','연제구','수영구','사상구','기장군']]
# pivoted.to_excel(OUTPUT_PATH + "1_허가연도_구군_건축물 허가 연면적 현황.xlsx", engine="openpyxl", index=False)

# 역세권 허가연면적 합계
def station_hub_baseinfo(gdf1, gdf2):
    sjoined = gpd.sjoin(gdf2, gdf1, how="inner", predicate="intersects")
    pivoted = sjoined.pivot_table(index=['허가연도','역세권'], columns='역명', values='연면적(㎡)', aggfunc='sum', fill_value=0).round(3).reset_index()
    return pivoted

sector_hub_baseinfo(gbase,ten_sectors).to_excel(OUTPUT_PATH + "1_허가연도_10대 중심지 생활권_건축물 허가 연면적 현황.xlsx", engine="openpyxl", index=False)
station_hub_baseinfo(gbase,역세권350).to_excel(OUTPUT_PATH + "1_허가연도_역세권350_건축물 허가 연면적 현황.xlsx", engine="openpyxl", index=False)
station_hub_baseinfo(gbase,역세권500).to_excel(OUTPUT_PATH + "1_허가연도_역세권500_건축물 허가 연면적 현황.xlsx", engine="openpyxl", index=False)

4. 건축물용도별 허가 연면적

In [42]:
buld_type = {
    '주거': ['공동주택', '단독주택','다가구주택','다세대주택'],
    '근생': ['근린생활시설','제1종근린생활시설'],
    '식음': ['제2종근린생활시설','위락시설'],
    '문화/교육/복지(의료)': ['문화및집회시설', '교육연구시설', '복지시설', '교육연구및복지시설', '의료시설','노유자시설'],
    '숙박': ['숙박시설'], '판매': ['판매시설','판매및영업시설'], '업무': ['업무시설','사무소'],
    '기타': ['공장', '제조업소', '창고', '종교', '자동차관련시설', '운수시설', '위험물저장및처리시설', '운동시설', 
             '수련시설', '관광휴게시설', '동식물관련시설', '분뇨·쓰레기처리시설', '교정및군사시설', '방소통신시설', 
	         '장례식장', '발전시설', '공공용시설', '묘지관련시설']}
name_to_group = {name: group for group, names in buld_type.items() for name in names}

gbase['구분'] = gbase['주용도코드명'].map(name_to_group)
gbase['구분'].fillna('기타', inplace=True)

# 10대 중심지 생활권 허가연면적 합계
def sector_hub_baseinfo(gdf1, gdf2):
    sjoined = gpd.sjoin(gdf2, gdf1, how="inner", predicate="intersects")
    pivoted = sjoined.pivot_table(index=['허가연도','구분'], columns='CNAME', values='연면적(㎡)', aggfunc='sum', fill_value=0).round(3).reset_index()
    return pivoted

# 구군별 허가연면적 합계
pivoted = gbase.pivot_table(index=['허가연도','구분'], columns='시군구명', values='연면적(㎡)', aggfunc='sum', fill_value=0).round(3).reset_index()
pivoted = pivoted[['허가연도','구분','중구','서구','동구','영도구','부산진구','동래구','남구','북구',
                         '해운대구','사하구','금정구','강서구','연제구','수영구','사상구','기장군']]
pivoted.to_excel(OUTPUT_PATH + "1_용도구분_구군_건축물 허가 연면적 현황.xlsx", engine="openpyxl", index=False)

# 역세권 허가연면적 합계
def station_hub_baseinfo(gdf1, gdf2):
    sjoined = gpd.sjoin(gdf2, gdf1, how="inner", predicate="intersects")
    pivoted = sjoined.pivot_table(index=['허가연도','구분','역세권'], columns='역명', values='연면적(㎡)', aggfunc='sum', fill_value=0).round(3).reset_index()
    return pivoted

sector_hub_baseinfo(gbase,ten_sectors).to_excel(OUTPUT_PATH + "1_용도구분_10대 중심지 생활권_건축물 허가 연면적 현황.xlsx", engine="openpyxl", index=False)
station_hub_baseinfo(gbase,역세권350).to_excel(OUTPUT_PATH + "1_용도구분_역세권350_건축물 허가 연면적 현황.xlsx", engine="openpyxl", index=False)
station_hub_baseinfo(gbase,역세권500).to_excel(OUTPUT_PATH + "1_용도구분_역세권500_건축물 허가 연면적 현황.xlsx", engine="openpyxl", index=False)


#### 3. 교통·유동인구
1. 대중교통(버스, 지하철 승하차) 이용인구

In [3]:
# 노선별 정류장위치정보 데이터 -> 시내와 마을로 분리 -> 연도별로 합치기
import pandas as pd
import geopandas as gpd
import glob

GPKG_PATH = r"2026-018_도시계획과.gpkg"
DATA_PATH = r"02_자료수집\\03_교통유동인구\\"
bus_lists = glob.glob(DATA_PATH + r"버스정류장정보\\*.xls", recursive=True)

values = []
for year in range(2019, 2026):
    for bus in bus_lists:
        if str(year) in bus:
            df = pd.read_excel(bus, dtype={'정류장코드': str}, engine="xlrd")
            values.append(df)
    concated = pd.concat(values, ignore_index=True)
    concated = concated[['운영기관','정류장코드','정류장명','경도좌표','위도좌표']]
    concated = gpd.GeoDataFrame(concated, 
                                               geometry=gpd.points_from_xy(concated['경도좌표'], concated['위도좌표']), 
                                               crs=4326).to_crs(5179)
    concated['X'] = concated.geometry.x.round(3)
    concated['Y'] = concated.geometry.y.round(3)
    concated.drop(columns=['경도좌표','위도좌표'], inplace=True)
    concated.to_file(GPKG_PATH, layer=f"정류장위치정보_{year}년", driver="GPKG")

In [ ]:
# 버스 이용인구 현황
# 부산빅데이터혁신센터에서 수집 중인 데이터 수령하여 작업
# 저장공간이 부족하여 연도-분기단위로 집계처리

import glob
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = r"02_자료수집\\03_교통유동인구\\01_버스\\"
GPKG_PATH = r"2026-018_도시계획과.gpkg"
OUTPUT_PATH = r"03_집계결과\\"
# months = [f"{y}{month:02d}" for y in range(2022, 2023) for month in range(1, 5)]
ten_sectors = gpd.read_file(GPKG_PATH, layer="10개_중심지_생활권", driver="GPKG")
gugun = gpd.read_file(GPKG_PATH, layer="NGII_SIG_26", driver="GPKG")
역세권350 = gpd.read_file(GPKG_PATH, layer="역세권350_74EA", driver="GPKG")
역세권500 = gpd.read_file(GPKG_PATH, layer="역세권500_58EA", driver="GPKG")

# 집계처리 함수생성
def sector_bustation(gdf2):
    print("2022년 1월 교통카드(버스) 이용자 현황집계 시작")
    values = []
    data_lists = glob.glob(DATA_PATH + rf"202201\\*.csv", recursive=True)
    for data in data_lists:
        gdf1 = gpd.read_file(GPKG_PATH, layer=f"정류장위치정보_2022년", driver="GPKG")
        df = pd.read_csv(data, dtype={'행데이터기준일자': str, '정류장코드': str, '탑승인원': int}, encoding="utf-8")
        merged = pd.merge(df, gdf1, on='정류장코드', how='left')
        merged = gpd.GeoDataFrame(merged, geometry=gpd.points_from_xy(merged['X'], merged['Y']), crs=5179)
        values.append(gpd.sjoin(gdf2, merged, how="inner", predicate="intersects"))
    concated = pd.concat(values, ignore_index=True)
    concated['기준연도'] = concated['행데이터기준일자'].str[:4] + "년"
    pivoted = pd.pivot_table(concated, index=['기준연도','운영기관'], columns='CNAME', 
                                         values='탑승인원', aggfunc='sum', fill_value=0).reset_index()
    print("2022년 1분기 교통카드(버스) 이용자 현황집계 종료")
    return pivoted

sector_bustation(ten_sectors).to_excel(OUTPUT_PATH + "3_10개 중심지 생활권_버스 이용인구 현황_202201.xlsx", 
                                       engine="openpyxl", index=False)


2022년 1분기 교통카드(버스) 이용자 현황집계 시작


  0%|          | 0/4 [18:14<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# 지하철 승하차 인원 현황

import pandas as pd
import geopandas as gpd
pd.set_option("display.max_rows", None)
import warnings
warnings.filterwarnings('ignore')

GPKG_PATH = r"02_자료수집\\2026-018_도시계획과.gpkg"
OUTPUT_PATH = r"03_집계결과\\"
ten_sectors = gpd.read_file(GPKG_PATH, layer="10개_중심지_생활권")
gugun = gpd.read_file(GPKG_PATH, layer="NGII_SIG_26")
역세권350 = gpd.read_file(GPKG_PATH, layer="역세권350_74EA")
역세권500 = gpd.read_file(GPKG_PATH, layer="역세권500_58EA")

# 10개 중심지 생활권 별 지하철 승하차인원 현황
def sectors_subway(gdf2):
    values = []
    for year in range(2019, 2026):
        gdf1 = gpd.read_file(GPKG_PATH, layer=f"도시철도시간대승하차인원{year}년")
        gdf1['기준연도'] = gdf1['년월일'].str[:4] + "년"
        values.append(gpd.sjoin(gdf2, gdf1, how="inner", predicate="intersects"))
    concated = pd.concat(values, ignore_index=True)
    pivoted = concated.pivot_table(index=['기준연도','역명'], columns='CNAME', values='합계', aggfunc='sum', fill_value=0).reset_index()
    return pivoted

# 부산시 전체(구군별) 지하철 승하차인원 현황
def gugun_subway(gdf2):
    values = []
    for year in range(2019, 2026):
        gdf1 = gpd.read_file(GPKG_PATH, layer=f"도시철도시간대승하차인원{year}년")
        gdf1['기준연도'] = gdf1['년월일'].str[:4] + "년"
        values.append(gpd.sjoin(gdf2, gdf1, how="inner", predicate="intersects"))
    concated = pd.concat(values, ignore_index=True)
    pivoted = concated.pivot_table(index=['기준연도','역명'], columns='SIG_KOR_NM', values='합계', 
                                         aggfunc='sum', fill_value=0).reset_index()
    pivoted = pivoted[['기준연도','역명','중구','서구','동구','부산진구','동래구','남구','북구',
                             '해운대구','사하구','금정구','강서구','연제구','수영구','사상구','기장군']]
    return pivoted

# 132개 역세권별 지하철 승하차인원 현황
def station_influence_area_subway(gdf2):
    values = []
    for year in range(2019, 2026):
        gdf1 = gpd.read_file(GPKG_PATH, layer=f"도시철도시간대승하차인원{year}년")
        gdf1['기준연도'] = gdf1['년월일'].str[:4] + "년"
        values.append(gpd.sjoin(gdf2, gdf1, how="inner", predicate="intersects"))
    concated = pd.concat(values, ignore_index=True)
    pivoted = concated.pivot_table(index=['기준연도','역세권'], columns='역명_right', values='합계', aggfunc='sum', fill_value=0).reset_index()
    pivoted.rename(columns={'역명_right': '역명'}, inplace=True)
    return pivoted

# sectors_subway(ten_sectors).to_excel(OUTPUT_PATH + "3_기준연도_10개 중심지 생활권_지하철_이용인구 현황.xlsx", engine="openpyxl", index=False)
# gugun_subway(gugun).to_excel(OUTPUT_PATH + "3_기준연도_구군_지하철_이용인구 현황.xlsx", engine="openpyxl", index=False)
# station_influence_area_subway(역세권350).to_excel(OUTPUT_PATH + "3_기준연도_역세권350_지하철_이용인구 현황.xlsx", engine="openpyxl", index=False)
station_influence_area_subway(역세권500).to_excel(OUTPUT_PATH + "3_기준연도_역세권500_지하철_이용인구 현황.xlsx", engine="openpyxl", index=False)

2. 유동인구 → 생활인구로 대체

In [ ]:
import glob
import pandas as pd
import geopandas as gpd
from impala.dbapi import connect
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import calendar
results = []
for year in range(2019, 2026):
    for month in range(1, 13):
        std_ym = str(year) + f"{month:02d}"
        daycnt = calendar.monthrange(year, month)[1]
        results.append({'기준연월': std_ym, '월별일수': daycnt})
monthly = pd.DataFrame(results)
monthly['기준연도'] = monthly['기준연월'].str[:4] + "년"
yearly = monthly.groupby('기준연도')['월별일수'].sum().reset_index(name="연별일수")

GPKG_PATH = r"02_자료수집\\2026-018_도시계획과.gpkg"
DATA_PATH = r"02_자료수집\\03_교통유동인구\\"
POP_PATH = r"D:\\02_사업관리\\자체분석\\01_생활인구\\"
FILES_PATH = r"D:\\02_사업관리\\자체분석\\04_공간DB\\griddata\\"
OUTPUT_PATH = r"03_집계결과\\"
conn = connect(host="172.20.192.154", port=10000, user="datawave", database="purchase_data", auth_mechanism="PLAIN")
pop_lists = glob.glob(POP_PATH + r"*_tme_p_*.csv", recursive=True)
months1 = [f"{y}{month:02d}" for y in range(2019, 2023) for month in range(1, 13)]
months2 = [f"{y}{month:02d}" for y in range(2023, 2026) for month in range(1, 13)]
ten_sectors = gpd.read_file(GPKG_PATH, layer="10개_중심지_생활권")
gugun = gpd.read_file(GPKG_PATH, layer="NGII_SIG_26")
역세권350 = gpd.read_file(GPKG_PATH, layer="역세권350_74EA")
역세권500 = gpd.read_file(GPKG_PATH, layer="역세권500_58EA")
grid50_pg = gpd.read_file(FILES_PATH + "busan_50cell_v15.shp", crs=5179)
grid50_pg['x'] = grid50_pg.geometry.centroid.x.round(3)
grid50_pg['y'] = grid50_pg.geometry.centroid.y.round(3)
grid50_pt = gpd.GeoDataFrame(grid50_pg, geometry=gpd.points_from_xy(grid50_pg.x, grid50_pg.y))
grid100_pg = gpd.read_file(FILES_PATH + "ngii_100m_busan.shp", crs=5179)
grid100_pg['x'] = grid100_pg.geometry.centroid.x.round(3)
grid100_pg['y'] = grid100_pg.geometry.centroid.y.round(3)
grid100_pt = gpd.GeoDataFrame(grid100_pg, geometry=gpd.points_from_xy(grid100_pg.x, grid100_pg.y))
ranges = ['심야2','오전1','오전2','오후1','오후2','심야1']
hour_to_range = {h: ranges[h // 4] for h in range(24)}

# 10개 중심지 생활권별 생활인구(2021년~2022년)
def sectors_lifepop(conn, grid1, gdf2):
    values = []
    for ym in tqdm(months1):
        query = f"""SELECT * FROM purchase_data.l0csv_kt_d_lifepop_time_p WHERE SUBSTR(std_ymd, 1, 6) = '{ym}'"""
        df = pd.read_sql(query, conn)
        df.columns = [col.split('.')[1] for col in df.columns]
        df.drop(['dat_flow_id','part_batchdate'], axis=1, inplace=True)
        cols = [col for col in df.columns if '_t_' in col]
        df[cols] = df[cols].astype(float)
        df['기준연도'] = df['std_ymd'].str[:4] + "년"
        df = pd.merge(df, grid1, on="id", how="inner")
        gdf = gpd.GeoDataFrame(df, geometry="geometry")
        gdf['주거'] = gdf.loc[:, 'h_t_00':'h_t_23'].sum(axis=1) / 24
        gdf['직장'] = gdf.loc[:, 'w_t_00':'w_t_23'].sum(axis=1) / 24
        gdf['방문'] = gdf.loc[:, 'v_t_00':'v_t_23'].sum(axis=1) / 24
        gdf['생활'] = gdf['주거'] + gdf['직장'] + gdf['방문']
        sjoined = gpd.sjoin(gdf2, gdf, how="inner", predicate="intersects")
        values.append(sjoined.pivot_table(index='기준연도', columns='CNAME', values='생활', aggfunc='sum', fill_value=0).round().reset_index())
    concated = pd.concat(values, ignore_index=True)
    conn.close()
    return concated

# 10개 중심지 생활권 생활인구(2023년~2025년)
def sectors_livingpop(conn, grid1, gdf2):
    values = []
    for ym in tqdm(months2):
        query = f"""SELECT * FROM purchase_data.l0kte_kt_d_living_tme_p2 WHERE SUBSTR(std_ymd, 1, 6) = '{ym}'"""
        df = pd.read_sql(query, conn)
        df.columns = [col.split('.')[1] for col in df.columns]
        df.drop(['dat_flow_id','part_batchdate'], axis=1, inplace=True)
        cols = [col for col in df.columns if '_t_' in col]
        df[cols] = df[cols].astype(float)
        df['기준연도'] = df['std_ymd'].str[:4] + "년"
        df = pd.merge(df, grid1, left_on="id", right_on="gid", how="inner")
        gdf = gpd.GeoDataFrame(df, geometry="geometry")
        gdf['주거'] = gdf.loc[:, 'h_t_00':'h_t_23'].sum(axis=1) / 24
        gdf['직장'] = gdf.loc[:, 'w_t_00':'w_t_23'].sum(axis=1) / 24
        gdf['방문'] = gdf.loc[:, 'v_t_00':'v_t_23'].sum(axis=1) / 24
        gdf['생활'] = gdf['주거'] + gdf['직장'] + gdf['방문']
        sjoined = gpd.sjoin(gdf2, gdf, how="inner", predicate="intersects")
        values.append(sjoined.pivot_table(index='기준연도', columns='CNAME', values='생활', aggfunc='sum', fill_value=0).round().reset_index())
    concated = pd.concat(values, ignore_index=True)
    conn.close()
    return concated

# 구군별 생활인구(2021년~2025년)
time1 = pd.read_csv(DATA_PATH + r"l0csv_kt_d_lifepop_time_2021-2022.csv", sep="|", dtype={'std_ymd': str, 'time': str, 'hcode': str}, encoding="utf-8")
time2 = pd.read_csv(DATA_PATH + r"l0kte_kt_d_living_tme_2023-2025.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'tme': str}, encoding="utf-8")
time1 = time1[['std_ymd','hcode','time','h_pop','w_pop','v_pop']]
time1.rename(columns={'time': 'tme'}, inplace=True)
time = pd.concat([time1, time2], ignore_index=True)
time['기준연도'] = time['std_ymd'].str[:4] + "년"
time['SIG_CD'] = time['hcode'].str[:5]
time = pd.merge(time, gugun[['SIG_CD','SIG_KOR_NM']], on="SIG_CD", how="inner")
time['시간'] = time['tme'] + "시"
time['시간대'] = time['tme'].astype(int).map(hour_to_range)
time['생활(일합계)'] = time.loc[:, 'h_pop': 'v_pop'].sum(axis=1)
times = time.groupby(['기준연도','SIG_CD','시간','시간대','SIG_KOR_NM'])['생활(일합계)'].sum().round().reset_index(name='생활(연합계)').sort_values(['기준연도','SIG_CD'], ascending=True)
times.rename(columns={'SIG_CD': '구군코드', 'SIG_KOR_NM': '구군명'}, inplace=True)
times = times.merge(yearly, on="기준연도", how="left")
times['생활(연평균)'] = (times['생활(연합계)'] / times['연별일수']).round()
times = times.pivot_table(index=['기준연도','시간','시간대'], columns='구군명', values='생활(연평균)').reset_index()
times = times[['기준연도','시간','시간대','중구','서구','동구','영도구','부산진구','동래구','남구',
                     '북구','해운대구','사하구','금정구','강서구','연제구','수영구','사상구','기장군']]
times.to_excel(OUTPUT_PATH + r"3_기준연도_구군_생활인구 현황.xlsx", engine="openpyxl", index=False)

# 역세권 생활인구(2019년~2022년)
def station_lifepop(conn, grid1, gdf2):
    values = []
    for ym in tqdm(months1):
        query = f"""SELECT * FROM purchase_data.l0csv_kt_d_lifepop_time_p WHERE SUBSTR(std_ymd, 1, 6) = '{ym}'"""
        df = pd.read_sql(query, conn)
        df.columns = [col.split('.')[1] for col in df.columns]
        df.drop(['dat_flow_id','part_batchdate'], axis=1, inplace=True)
        cols = [col for col in df.columns if '_t_' in col]
        df[cols] = df[cols].astype(float)
        df['기준연도'] = df['std_ymd'].str[:4] + "년"
        df = pd.merge(df, grid1, on="id", how="inner")
        gdf = gpd.GeoDataFrame(df, geometry="geometry")
        gdf['주거'] = gdf.loc[:, 'h_t_00':'h_t_23'].sum(axis=1) / 24
        gdf['직장'] = gdf.loc[:, 'w_t_00':'w_t_23'].sum(axis=1) / 24
        gdf['방문'] = gdf.loc[:, 'v_t_00':'v_t_23'].sum(axis=1) / 24
        gdf['생활(일합계)'] = gdf['주거'] + gdf['직장'] + gdf['방문']
        sjoined = gpd.sjoin(gdf2, gdf, how="inner", predicate="intersects")
        values.append(sjoined.groupby(['기준연도','역세권','역명']
                                      )['생활(일합계)'].sum().round().reset_index(name="생활(연합계)").sort_values(['기준연도'], ascending=True))
    concated = pd.concat(values, ignore_index=True)
    pivoted = concated.pivot_table(index=['기준연도','역세권'], columns='역명', values='생활(연합계)').reset_index()
    conn.close()
    return pivoted

# 역세권 생활인구(2023년~2025년)
def station_livingpop(conn, grid1, gdf2):
    values = []
    for ym in tqdm(months2):
        query = f"""SELECT * FROM purchase_data.l0kte_kt_d_living_tme_p2 WHERE SUBSTR(std_ymd, 1, 6) = '{ym}'"""
        df = pd.read_sql(query, conn)
        df.columns = [col.split('.')[1] for col in df.columns]
        df.drop(['dat_flow_id','part_batchdate'], axis=1, inplace=True)
        cols = [col for col in df.columns if '_t_' in col]
        df[cols] = df[cols].astype(float)
        df['기준연도'] = df['std_ymd'].str[:4] + "년"
        df = pd.merge(df, grid1, left_on="id", right_on="gid", how="inner")
        gdf = gpd.GeoDataFrame(df, geometry="geometry")
        gdf['주거'] = gdf.loc[:, 'h_t_00':'h_t_23'].sum(axis=1) / 24
        gdf['직장'] = gdf.loc[:, 'w_t_00':'w_t_23'].sum(axis=1) / 24
        gdf['방문'] = gdf.loc[:, 'v_t_00':'v_t_23'].sum(axis=1) / 24
        gdf['생활(일합계)'] = gdf['주거'] + gdf['직장'] + gdf['방문']
        sjoined = gpd.sjoin(gdf2, gdf, how="inner", predicate="intersects")
        values.append(sjoined.groupby(['기준연도','역세권','역명']
                                      )['생활(일합계)'].sum().round().reset_index(name='생활(연합계)').sort_values(['기준연도'], ascending=True))
    concated = pd.concat(values, ignore_index=True)
    pivoted = concated.pivot_table(index=['기준연도','역세권'], columns='역명', values='생활(연합계)').reset_index()
    conn.close()
    return pivoted

# pd.concat([sectors_lifepop(conn, grid50_pt, ten_sectors),
#            sectors_livingpop(conn, grid100_pt, ten_sectors)],
#           ignore_index=True).to_excel(OUTPUT_PATH + "3_기준연도_10개 중심지 생활권_생활인구 현황.xlsx", engine="openpyxl", index=False)

pd.concat([station_lifepop(conn, grid50_pt, 역세권350),
           station_livingpop(conn, grid100_pt, 역세권350)],
          ignore_index=True).to_excel(OUTPUT_PATH + "3_기준연도_역세권350_생활인구 현황.xlsx", engine="openpyxl", index=False)

# pd.concat([station_lifepop(conn, grid50_pt, 역세권500),
#            station_livingpop(conn, grid100_pt, 역세권500)],
#           ignore_index=True).to_excel(OUTPUT_PATH + "3_기준연도_역세권500_생활인구 현황.xlsx", engine="openpyxl", index=False)


100%|██████████| 1/1 [07:31<00:00, 451.65s/it]


In [4]:
# 역세권 생활인구(2023년~2025년)
def station_livingpop(conn, grid1, gdf2):
    values = []
    for ym in tqdm(months2):
        query = f"""SELECT * FROM purchase_data.l0kte_kt_d_living_tme_p2 WHERE SUBSTR(std_ymd, 1, 6) = '{ym}'"""
        df = pd.read_sql(query, conn)
        df.columns = [col.split('.')[1] for col in df.columns]
        df.drop(['dat_flow_id','part_batchdate'], axis=1, inplace=True)
        cols = [col for col in df.columns if '_t_' in col]
        df[cols] = df[cols].astype(float)
        df['기준연도'] = df['std_ymd'].str[:4] + "년"
        df = pd.merge(df, grid1, left_on="id", right_on="gid", how="inner")
        gdf = gpd.GeoDataFrame(df, geometry="geometry")
        gdf['주거'] = gdf.loc[:, 'h_t_00':'h_t_23'].sum(axis=1) / 24
        gdf['직장'] = gdf.loc[:, 'w_t_00':'w_t_23'].sum(axis=1) / 24
        gdf['방문'] = gdf.loc[:, 'v_t_00':'v_t_23'].sum(axis=1) / 24
        gdf['생활'] = gdf['주거'] + gdf['직장'] + gdf['방문']
        sjoined = gpd.sjoin(gdf2, gdf, how="inner", predicate="intersects")
        values.append(sjoined.groupby(['기준연도','역세권','역명'], as_index=False)['생활'].sum().round().reset_index().sort_values(['기준연도'], ascending=True))
    concated = pd.concat(values, ignore_index=True)
    pivoted = concated.pivot_table(index=['기준연도','역세권'], columns='역명', values='생활', fill_value=0).reset_index()
    conn.close()
    return pivoted

,기준연도,연별일수
0,2019,365
1,2020,366
2,2021,365
3,2022,365
4,2023,365
5,2024,366
6,2025,365
